In [ ]:
print("\n" + "="*80)
print("ALGORITHM 1: K-MEANS CLUSTERING")
print("="*80)

start_time = time.time()

k_values = [3, 5, 7, 10]
kmeans_results = []

for k in k_values:
    print(f"\nTraining K-Means with k={k}...")

    kmeans = KMeans(
        featuresCol="features",
        predictionCol=f"cluster_k{k}",
        k=k,
        seed=42,
        maxIter=20,
        initMode="k-means||"
    )

    kmeans_model = kmeans.fit(train_df)

    # Predictions
    train_pred = kmeans_model.transform(train_df)
    val_pred = kmeans_model.transform(val_df)

    # Evaluate
    evaluator = ClusteringEvaluator(
        featuresCol="features",
        predictionCol=f"cluster_k{k}",
        metricName="silhouette"
    )

    train_silhouette = evaluator.evaluate(train_pred)
    val_silhouette = evaluator.evaluate(val_pred)
    wssse = kmeans_model.summary.trainingCost

    kmeans_results.append({
        'k': k,
        'train_silhouette': train_silhouette,
        'val_silhouette': val_silhouette,
        'wssse': wssse,
        'iterations': kmeans_model.summary.numIter
    })

    print(f"  Train Silhouette: {train_silhouette:.4f}")
    print(f"  Val Silhouette:   {val_silhouette:.4f}")
    print(f"  WSSSE: {wssse:.2f}")

# Best k
best_k = __builtins__.max(kmeans_results, key=lambda x: x['val_silhouette'])
print(f"\n Best k={best_k['k']} (Validation Silhouette: {best_k['val_silhouette']:.4f})")

# Train final model
kmeans_final = KMeans(
    featuresCol="features",
    predictionCol="kmeans_cluster",
    k=best_k['k'],
    seed=42,
    maxIter=20
)
kmeans_final_model = kmeans_final.fit(train_df)

# Transform all datasets
train_df = kmeans_final_model.transform(train_df)
val_df = kmeans_final_model.transform(val_df)
test_df = kmeans_final_model.transform(test_df)

# Test evaluation
test_silhouette = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="kmeans_cluster",
    metricName="silhouette"
).evaluate(test_df)

print(f"Test Silhouette: {test_silhouette:.4f}")

# Save results
pd.DataFrame(kmeans_results).to_csv('kmeans_results.csv', index=False)

kmeans_time = time.time() - start_time
print(f"\n K-Means completed in {kmeans_time:.2f}s")

In [ ]:
print("\n" + "="*80)
print("ALGORITHM 2: BISECTING K-MEANS")
print("="*80)

start_time = time.time()

bkmeans_results = []

for k in k_values:
    print(f"\nTraining Bisecting K-Means with k={k}...")

    bkmeans = BisectingKMeans(
        featuresCol="features",
        predictionCol=f"bkmeans_cluster_k{k}",
        k=k,
        seed=42,
        maxIter=20
    )

    bkmeans_model = bkmeans.fit(train_df)
    train_pred = bkmeans_model.transform(train_df)
    val_pred = bkmeans_model.transform(val_df)

    evaluator = ClusteringEvaluator(
        featuresCol="features",
        predictionCol=f"bkmeans_cluster_k{k}",
        metricName="silhouette"
    )

    train_silhouette = evaluator.evaluate(train_pred)
    val_silhouette = evaluator.evaluate(val_pred)
    wssse = bkmeans_model.summary.trainingCost

    bkmeans_results.append({
        'k': k,
        'train_silhouette': train_silhouette,
        'val_silhouette': val_silhouette,
        'wssse': wssse
    })

    print(f"  Train Silhouette: {train_silhouette:.4f}")
    print(f"  Val Silhouette:   {val_silhouette:.4f}")

best_k_bkmeans = __builtins__.max(bkmeans_results, key=lambda x: x['val_silhouette'])
print(f"\n Best k={best_k_bkmeans['k']} (Validation Silhouette: {best_k_bkmeans['val_silhouette']:.4f})")

# Final model
bkmeans_final = BisectingKMeans(
    featuresCol="features",
    predictionCol="bkmeans_cluster",
    k=best_k_bkmeans['k'],
    seed=42,
    maxIter=20
)
bkmeans_final_model = bkmeans_final.fit(train_df)

train_df = bkmeans_final_model.transform(train_df)
val_df = bkmeans_final_model.transform(val_df)
test_df = bkmeans_final_model.transform(test_df)

test_silhouette_bkmeans = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="bkmeans_cluster",
    metricName="silhouette"
).evaluate(test_df)

print(f"Test Silhouette: {test_silhouette_bkmeans:.4f}")

pd.DataFrame(bkmeans_results).to_csv('bkmeans_results.csv', index=False)

bkmeans_time = time.time() - start_time
print(f"\n Bisecting K-Means completed in {bkmeans_time:.2f}s")

In [ ]:
print("\n" + "="*80)
print("ALGORITHM 3: GAUSSIAN MIXTURE MODEL")
print("="*80)

start_time = time.time()

gmm_results = []

for k in k_values:
    print(f"\nTraining GMM with k={k}...")

    gmm = GaussianMixture(
        featuresCol="features",
        predictionCol=f"gmm_cluster_k{k}",
        k=k,
        seed=42,
        maxIter=20
    )

    gmm_model = gmm.fit(train_df)
    train_pred = gmm_model.transform(train_df)
    val_pred = gmm_model.transform(val_df)

    evaluator = ClusteringEvaluator(
        featuresCol="features",
        predictionCol=f"gmm_cluster_k{k}",
        metricName="silhouette"
    )

    train_silhouette = evaluator.evaluate(train_pred)
    val_silhouette = evaluator.evaluate(val_pred)
    log_likelihood = gmm_model.summary.logLikelihood

    gmm_results.append({
        'k': k,
        'train_silhouette': train_silhouette,
        'val_silhouette': val_silhouette,
        'log_likelihood': log_likelihood
    })

    print(f"  Train Silhouette: {train_silhouette:.4f}")
    print(f"  Val Silhouette:   {val_silhouette:.4f}")

best_k_gmm = __builtins__.max(gmm_results, key=lambda x: x['val_silhouette'])
print(f"\n Best k={best_k_gmm['k']} (Validation Silhouette: {best_k_gmm['val_silhouette']:.4f})")

# Final model
gmm_final = GaussianMixture(
    featuresCol="features",
    predictionCol="gmm_cluster",
    k=best_k_gmm['k'],
    seed=42,
    maxIter=20
)
gmm_final_model = gmm_final.fit(train_df)

train_df = gmm_final_model.transform(train_df)
val_df = gmm_final_model.transform(val_df)
test_df = gmm_final_model.transform(test_df)

test_silhouette_gmm = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="gmm_cluster",
    metricName="silhouette"
).evaluate(test_df)

print(f"Test Silhouette: {test_silhouette_gmm:.4f}")

pd.DataFrame(gmm_results).to_csv('gmm_results.csv', index=False)

gmm_time = time.time() - start_time
print(f"\n  GMM completed in {gmm_time:.2f}s")

In [ ]:
print("\n" + "="*80)
print("ALGORITHM 4: RANDOM FOREST CLASSIFIER")
print("="*80)

start_time = time.time()

print("Training Random Forest for length category prediction...")

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="length_category_idx",
    predictionCol="rf_prediction",
    rawPredictionCol="rf_raw_prediction", # Added to avoid column naming conflicts
    probabilityCol="rf_probability",      # Added to avoid column naming conflicts
    numTrees=50,
    maxDepth=10,
    seed=42,
    featureSubsetStrategy="auto",
    subsamplingRate=0.8
)

rf_model = rf.fit(train_df)

# Predictions
train_pred_rf = rf_model.transform(train_df)
val_pred_rf = rf_model.transform(val_df)
test_pred_rf = rf_model.transform(test_df)

# Evaluate
evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="length_category_idx",
    predictionCol="rf_prediction",
    metricName="accuracy"
)

evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="length_category_idx",
    predictionCol="rf_prediction",
    metricName="f1"
)

train_accuracy = evaluator_acc.evaluate(train_pred_rf)
val_accuracy = evaluator_acc.evaluate(val_pred_rf)
test_accuracy = evaluator_acc.evaluate(test_pred_rf)

train_f1 = evaluator_f1.evaluate(train_pred_rf)
val_f1 = evaluator_f1.evaluate(val_pred_rf)
test_f1 = evaluator_f1.evaluate(test_pred_rf)

print(f"\nAccuracy - Train: {train_accuracy:.4f}, Val: {val_accuracy:.4f}, Test: {test_accuracy:.4f}")
print(f"F1 Score - Train: {train_f1:.4f}, Val: {val_f1:.4f}, Test: {test_f1:.4f}")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': rf_model.featureImportances.toArray()
}).sort_values('importance', ascending=False)

print("\nTop 10 Important Features:")
print(feature_importance.head(10).to_string(index=False))

feature_importance.to_csv('rf_feature_importance.csv', index=False)

rf_time = time.time() - start_time
print(f"\n Random Forest completed in {rf_time:.2f}s")
train_df = train_pred_rf
val_df = val_pred_rf
test_df = test_pred_rf

In [ ]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol="length_category_idx",
    predictionCol="lr_prediction",
    probabilityCol="lr_probability",
    rawPredictionCol="lr_raw_prediction",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0,
    family="multinomial",
    standardization=True,
    aggregationDepth=2
)

lr_model = lr.fit(train_df)
print(f" Model trained in {lr_model.summary.totalIterations} iterations")